In [16]:
import os
import json 
import shutil
import nltk
from statistics import mean

In [17]:
def jaccard_similarity(output1, output2):
    set1 = set(output1)
    set2 = set(output2)
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union

def calculate_bleu(reference, hypothesis):
    reference = [reference]
    hypothesis = hypothesis
    reference_tokens = [nltk.word_tokenize(ref) for ref in reference]
    hypothesis_tokens = nltk.word_tokenize(hypothesis)
    # Berechne den BLEU-Score
    bleu_score = nltk.translate.bleu_score.sentence_bleu(reference_tokens, hypothesis_tokens)
    return bleu_score

In [42]:
def json_to_text(data):
    if isinstance(data, dict):
        if "AND" in data:
            left = json_to_text(data["AND"]["left"])
            right = json_to_text(data["AND"]["right"])
            return f"({left} AND {right})"
        elif "OR" in data:
            left = json_to_text(data["OR"]["left"])
            right = json_to_text(data["OR"]["right"])
            return f"({left} OR {right})"
        elif "NOT" in data:
            inner = json_to_text(data["NOT"]["left"])
            return f"(NOT {inner})"
        elif "raw_text" in data:
            return data["raw_text"]
    return ""
def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [43]:
model_name = "Llama-3-70B-Instruct_5_shot"
model_path = f"model_output/{model_name}/ready"
failed_inner_path = f"../model_output/{model_name}/failed_inner"
label_path = "../../chia_label/p2"
base_model_output_path = "model_output"

In [40]:
model_names = [d for d in os.listdir(base_model_output_path) if os.path.isdir(os.path.join(base_model_output_path, d))]

for model_name in model_names:
    model_path = os.path.join(base_model_output_path, model_name, "ready")
    failed_inner_path = os.path.join(base_model_output_path, model_name, "failed_inner")

    label_files = os.listdir(label_path)
    model_files = os.listdir(model_path)
    bleu_scores = []
    jaccard_similarities = []

    for label_file in label_files:
        base_name = label_file.split("_parsed_2.json")[0]

        # Extract model and nshot (taking into account that there might be two "temp" components)
        parts = model_name.split("_")
        model = "_".join(parts[:-2])
        nshot = parts[-2]
        model_file = f"{model}_{base_name}_{nshot}_shot.json"

        if model_file in model_files:
            label_file_path = os.path.join(label_path, label_file)
            model_file_path = os.path.join(model_path, model_file)

            label_data = read_json(label_file_path)
            #model_data = read_json(model_file_path)

            try:
                label_text = json_to_text(label_data)
                #model_text = json_to_text(model_data)

                #bleu_score = calculate_bleu(reference=label_text, hypothesis=model_text)
                #jaccard_score = jaccard_similarity(label_text, model_text)
                #bleu_scores.append(bleu_score)
                #jaccard_similarities.append(jaccard_score)

                #print(f"Label: {label_file}")
                #print(f"Model: {model_file}")
                #print(f"BLEU Score: {bleu_score}")
                #print(f"Jaccard Similarity: {jaccard_score}")

            except KeyError as e:
                print(f"Error processing file {label_file}: {e}")
                print(f"Error processing file {model_file}: {e}")
                failed_model_path = os.path.join(failed_inner_path, model_file)
                shutil.move(model_file_path, failed_model_path)

    # Durchschnittswerte berechnen
    if bleu_scores:
        average_bleu = mean(bleu_scores)
        average_jaccard = mean(jaccard_similarities)

        print(f"Durchschnittlicher BLEU Score für {model_name}: {average_bleu}")
        print(f"Durchschnittliche Jaccard Ähnlichkeit für {model_name}: {average_jaccard}")

(((((((((((((Patients with symptomatic CNS metastases OR or leptomeningeal involvement) AND (Patients with known brain metastases AND (, AND (NOT (unless these metastases have been treated OR (and/or have been stable for at least six months prior to study start. Subjects with a history of AND (brain metastases AND must have a head CT with contrast to document either response or progression.))))))) AND (Patients with bone metastases AND as the only site(s) of measurable disease)) AND (Patients with hepatic artery chemoembolization within the last 6 months OR (one month if there are other sites of measurable disease))) AND Patients who have been previously treated with radioactive directed therapies) AND Patients who have been previously treated with epothilone) AND (Patients with any peripheral neuropathy OR (or unresolved diarrhea AND greater than Grade 1))) AND (Patients with severe cardiac insufficiency AND (patients taking Coumadin OR (or other warfarin-containing agents AND (NOT (w

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [31]:
# Durchschnittswerte berechnen
average_bleu = mean(bleu_scores)
average_jaccard = mean(jaccard_similarities)

print(f"Durchschnittlicher BLEU Score: {average_bleu}")
print(f"Durchschnittliche Jaccard Ähnlichkeit: {average_jaccard}")

StatisticsError: mean requires at least one data point

In [30]:
# Label
label_text = json_to_text(label_data)
print(label_text)

(((Provide written informed consent before beginning any study related activities AND Be between age 18 and 55 years) AND Be able to speak, read and write English and follow simple instructions for completing self-rated scales) AND (Meet DSM-IV criteria for BPD AND as assessed by the Structured Clinical Interview for DSM-IV Personality Disorders (SCID-II).))


In [31]:
# Model
model_text = json_to_text(model_data)
print(model_text)

((Provide written informed consent before beginning any study related activities AND (Be between age 18 and 55 years AND (Be able to speak, read and write English AND and follow simple instructions for completing self-rated scales))) AND Meet DSM-IV criteria for BPD as assessed by the Structured Clinical Interview for DSM-IV Personality Disorders (SCID-II))


In [33]:
calculate_bleu(reference=label_text , hypothesis=model_text)

0.7726951981286995

In [12]:
def json_to_text(data):
    if isinstance(data, dict):
        if "AND" in data:
            left = json_to_text(data["AND"].get("left", {}))
            right = json_to_text(data["AND"].get("right", {}))
            return f"({left} AND {right})"
        elif "OR" in data:
            left = json_to_text(data["OR"].get("left", {}))
            right = json_to_text(data["OR"].get("right", {}))
            return f"({left} OR {right})"
        elif "NOT" in data:
            inner = json_to_text(data["NOT"].get("left", {}))
            return f"(NOT {inner})"
        elif "raw_text" in data:
            return data["raw_text"]
    return ""

In [14]:
json_to_text(data)

'((((((((Pathologically proven unresectable adenocarcinoma of stomach AND With uni-dimensionally measurable disease (at least longest diameter 2 cm on conventional CT scan, x-ray or physical examination, or 1cm on spiral CT scan)) AND (Age 18 to 70 years old AND Estimated life expectancy of more than 3 months)) AND ECOG performance status of 2 or lower) AND (Adequate bone marrow function(absolute neutrophil count [ANC] ≥1,500/µL, hemoglobin ≥9.0 g/dL,and platelets ≥100,000/µL) AND Adequate kidney function (serum creatinine < 1.5 mg/dL))) AND Adequate liver function (serum total bilirubin < 2 times the upper normal limit (UNL); serum transaminases levels <3 times [<5 times for patients with liver metastasis] UNL)) AND (NOT No prior chemotherapy but prior adjuvant chemotherapy finished at least 6 months before enrollment was allowed. (but, prior adjuvant chemotherapy with capecitabine or S-1 or camptothecin analogues was excluded))) AND No prior radiation therapy for at least 4 weeks bef

In [15]:
jaccard_similarity(label_text,model_text)

NameError: name 'label_text' is not defined